# RetailPulse: AI-Powered Sales & Retail Analytics Platform
## Day 27 — Load Testing & Final Accuracy Validation

**Module:** Production Readiness (Phase 5)

Two things: (1) re-validate every model's accuracy metric against its target one final time, on the current state of all artifacts; (2) report the Locust load test run against the live dashboard (executed directly from the terminal — see `load_testing/locustfile.py` and the raw CSV this notebook loads).

In [1]:
import json
import pandas as pd

MODELS_DIR = "../models"
LOAD_TEST_DIR = "../load_testing"

## 1. Final Accuracy Validation — All Models vs Targets

In [2]:
with open(f"{MODELS_DIR}/hybrid_metrics.json") as f:
    hybrid = json.load(f)
with open(f"{MODELS_DIR}/churn_metrics_tuned.json") as f:
    churn = json.load(f)

validation = pd.DataFrame([
    {"Model": "Hybrid Forecast (Prophet+LSTM)", "Metric": "MAPE", "Target": "<= 12%",
     "Actual": f"{hybrid['mape']:.2f}%", "Pass": hybrid["mape"] <= 12},
    {"Model": "Churn XGBoost (tuned)", "Metric": "AUC-ROC", "Target": ">= 0.88",
     "Actual": f"{churn['auc_roc']:.4f}", "Pass": churn["auc_roc"] >= 0.88},
    {"Model": "Churn XGBoost (tuned)", "Metric": "Precision@Top-20%", "Target": ">= 0.75",
     "Actual": f"{churn['precision_at_top20']:.4f}", "Pass": churn["precision_at_top20"] >= 0.75},
])

validation

,Model,Metric,Target,Actual,Pass
0,Hybrid Forecast (Prophet+LSTM),MAPE,<= 12%,6.26%,True
1,Churn XGBoost (tuned),AUC-ROC,>= 0.88,0.9694,True
2,Churn XGBoost (tuned),Precision@Top-20%,>= 0.75,1.0000,True


In [3]:
all_pass = validation["Pass"].all()
print(f"All acceptance criteria met: {all_pass}")
assert all_pass, "A model has regressed below its acceptance target — investigate before deployment."

All acceptance criteria met: True


## 2. Load Test Results

Locust, headless, 20 concurrent users ramping at 5/s, 30-second run against the live dashboard (`streamlit run Home.py`) on localhost. Hits the health endpoint plus four page routes.

In [4]:
load_results = pd.read_csv(f"{LOAD_TEST_DIR}/loadtest_results_stats.csv")
load_results[["Name", "Request Count", "Failure Count", "Median Response Time",
              "Average Response Time", "Requests/s", "95%", "99%"]]

,Name,Request Count,Failure Count,Median Response Time,Average Response Time,Requests/s,95%,99%
0,/ (Home),59,0,3,10.235198,1.969779,76,140
1,/Churn_Risk,59,0,3,7.900801,1.969779,40,140
2,/Demand_Forecast,41,0,3,7.513426,1.368829,16,130
3,/Inventory_Health,17,0,3,3.519103,0.567563,12,12
4,/_stcore/health,116,0,2,3.988053,3.872785,9,22
5,Aggregated,292,0,3,6.508607,9.748735,17,140


In [5]:
aggregated = load_results[load_results["Name"] == "Aggregated"].iloc[0]
print(f"Total requests:       {int(aggregated['Request Count'])}")
print(f"Failures:             {int(aggregated['Failure Count'])} "
      f"({aggregated['Failure Count'] / aggregated['Request Count'] * 100:.2f}%)")
print(f"Median response time: {aggregated['Median Response Time']:.0f} ms")
print(f"P95 response time:    {aggregated['95%']:.0f} ms")
print(f"P99 response time:    {aggregated['99%']:.0f} ms")
print(f"Throughput:           {aggregated['Requests/s']:.1f} req/s")

print(f"\nZero failures under load: {'YES' if aggregated['Failure Count'] == 0 else 'NO'}")

Total requests:       292
Failures:             0 (0.00%)
Median response time: 3 ms
P95 response time:    17 ms
P99 response time:    140 ms
Throughput:           9.7 req/s

Zero failures under load: YES


## 3. Day 27 Checkpoint Summary

**Outputs saved:**
- `load_testing/locustfile.py` — the load test script
- `load_testing/loadtest_results_stats.csv` — raw results from the executed run

**Result:** all three model acceptance criteria still pass on the current artifacts; the live dashboard held 292 requests across 5 endpoints with zero failures at 20 concurrent users (median 3ms, p95 17ms).

**Scope note:** this load test ran against `localhost` in the build environment, not a deployed cloud instance — network latency, real user concurrency, and infrastructure-level bottlenecks (load balancer, ingress, database connections) aren't captured here. Re-run against the actual deployed URL before relying on these numbers for capacity planning.

**Next module:** `28_final_qa` — README, submission packaging, final QA checklist.